# Notebook 04: Search and Lineage

**Objective:** Use full-text search across the catalog and explore table relationships and lineage.

**Prerequisites:**
- Completed Notebooks 01-03 (lab running, tables discovered, descriptions and tags added)

**What you will learn:**
1. Search for tables containing "customer" using full-text search
2. Search by tag (find all PII-tagged assets)
3. Explore table-level relationships (FK-based lineage)
4. Add a custom lineage edge via the API
5. Query downstream dependencies (impact analysis)

## Setup: Configuration and Authentication

In [ ]:
import requests
import json
import time

# OpenMetadata API base URL (internal Docker network)
OM_URL = "http://openmetadata-server:8585/api/v1"

def get_auth_token(base_url, username="admin", password="admin", max_retries=12, wait_seconds=15):
    """Authenticate with OpenMetadata and return JWT token."""
    login_url = f"{base_url}/users/login"
    payload = {"email": username, "password": password}
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(login_url, json=payload, timeout=10)
            if resp.status_code == 200:
                print(f"Authenticated successfully")
                return resp.json()["accessToken"]
            print(f"Attempt {attempt}/{max_retries}: HTTP {resp.status_code} - retrying...")
        except (requests.ConnectionError, requests.Timeout):
            print(f"Attempt {attempt}/{max_retries}: Server not ready - retrying...")
        time.sleep(wait_seconds)
    raise ConnectionError("Could not connect to OpenMetadata")

token = get_auth_token(OM_URL)
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
patch_headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json-patch+json"}

# Table FQN prefix
FQN_PREFIX = "ecommerce_postgres.ecommerce.public"

print("Ready for search and lineage exercises.")

## Exercise 1: Full-Text Search for "customer"

OpenMetadata provides full-text search across all metadata: table names, descriptions, column names, tags, and more. This is powered by Elasticsearch.

Let's search for everything related to "customer" in our catalog.

In [ ]:
# Full-text search for "customer"
resp = requests.get(
    f"{OM_URL}/search/query",
    headers=headers,
    params={
        "q": "customer",
        "from": 0,
        "size": 20
    }
)

if resp.status_code == 200:
    result = resp.json()
    total = result.get("hits", {}).get("total", {}).get("value", 0)
    hits = result.get("hits", {}).get("hits", [])
    
    print(f"Search results for 'customer': {total} matches")
    print("=" * 70)
    
    for hit in hits:
        source = hit.get("_source", {})
        entity_type = source.get("entityType", hit.get("_index", "unknown"))
        name = source.get("name", source.get("displayName", "N/A"))
        fqn = source.get("fullyQualifiedName", "N/A")
        desc = source.get("description", "No description")[:60]
        score = hit.get("_score", 0)
        
        print(f"\n  [{entity_type}] {name} (score: {score:.2f})")
        print(f"    FQN: {fqn}")
        print(f"    Description: {desc}")
else:
    print(f"Search failed: {resp.status_code} - {resp.text[:200]}")

## Exercise 2: Search by Tag (Find PII Assets)

Searching by tag is essential for governance. For example, during a privacy audit, you need to find all assets tagged as PII.

OpenMetadata search supports filtering by tag using the `query_filter` parameter.

In [ ]:
# Search for all assets tagged with PII using Elasticsearch query DSL
query_filter = {
    "query": {
        "bool": {
            "must": [
                {
                    "bool": {
                        "should": [
                            {"term": {"tags.tagFQN": "PII.Sensitive"}},
                            {"term": {"tags.tagFQN": "PII.NonSensitive"}},
                            {"term": {"columns.tags.tagFQN": "PII.Sensitive"}}
                        ]
                    }
                }
            ]
        }
    }
}

resp = requests.get(
    f"{OM_URL}/search/query",
    headers=headers,
    params={
        "q": "*",
        "index": "table_search_index",
        "query_filter": json.dumps(query_filter),
        "from": 0,
        "size": 50
    }
)

if resp.status_code == 200:
    result = resp.json()
    hits = result.get("hits", {}).get("hits", [])
    
    print(f"Assets with PII tags: {len(hits)}")
    print("=" * 70)
    
    for hit in hits:
        source = hit.get("_source", {})
        fqn = source.get("fullyQualifiedName", "N/A")
        table_tags = [t.get("tagFQN", "") for t in source.get("tags", [])]
        
        print(f"\nTable: {fqn}")
        if table_tags:
            print(f"  Table tags: {table_tags}")
        
        # Show PII columns
        for col in source.get("columns", []):
            col_tags = [t.get("tagFQN", "") for t in col.get("tags", [])]
            if any("PII" in tag for tag in col_tags):
                print(f"  Column '{col['name']}': {col_tags}")
else:
    print(f"Search failed: {resp.status_code}")
    print("Note: The exact query filter syntax may vary by OM version.")

## Exercise 3: Explore Table Relationships

Our e-commerce schema has natural foreign key relationships:

```
customers <-- orders (customer_id FK)
orders <-- order_items (order_id FK)
products <-- order_items (product_id FK)
```

OpenMetadata can discover these relationships during metadata ingestion. Let's explore them via the lineage API.

In [ ]:
# Get the table entity ID for the customers table
fqn = f"{FQN_PREFIX}.customers"
resp = requests.get(f"{OM_URL}/tables/name/{fqn}", headers=headers)
customers_table = resp.json()
customers_id = customers_table["id"]
print(f"Customers table ID: {customers_id}")

# Get lineage for the customers table
resp = requests.get(
    f"{OM_URL}/lineage/table/name/{fqn}",
    headers=headers,
    params={"upstreamDepth": 3, "downstreamDepth": 3}
)

if resp.status_code == 200:
    lineage = resp.json()
    
    # Parse the lineage graph
    entity = lineage.get("entity", {})
    upstream = lineage.get("upstreamEdges", [])
    downstream = lineage.get("downstreamEdges", [])
    nodes = {n["id"]: n.get("fullyQualifiedName", n.get("name", "unknown")) for n in lineage.get("nodes", [])}
    nodes[entity.get("id", "")] = entity.get("fullyQualifiedName", "root")
    
    print(f"\nLineage for: {entity.get('fullyQualifiedName', 'N/A')}")
    print(f"  Upstream edges: {len(upstream)}")
    print(f"  Downstream edges: {len(downstream)}")
    
    if upstream:
        print("\n  Upstream (data flows FROM):")
        for edge in upstream:
            from_id = edge.get("fromEntity", "")
            print(f"    <- {nodes.get(from_id, from_id)}")
    
    if downstream:
        print("\n  Downstream (data flows TO):")
        for edge in downstream:
            to_id = edge.get("toEntity", "")
            print(f"    -> {nodes.get(to_id, to_id)}")
    
    if not upstream and not downstream:
        print("\n  No lineage edges found.")
        print("  FK-based lineage may need to be added via ingestion or manually (see Exercise 4).")
else:
    print(f"Lineage request failed: {resp.status_code}")
    print("Lineage may not be populated yet. Proceed to Exercise 4 to add lineage manually.")

## Exercise 4: Add Custom Lineage Edge

Sometimes lineage is not automatically discovered (e.g., transformations in application code, ETL pipelines). In these cases, you can add lineage edges manually via the API.

Let's add lineage showing that `orders` depends on `customers` (the customer_id FK relationship).

In [ ]:
# Get table IDs for lineage edge creation
table_ids = {}
for table_name in ["customers", "products", "orders", "order_items"]:
    fqn = f"{FQN_PREFIX}.{table_name}"
    resp = requests.get(f"{OM_URL}/tables/name/{fqn}", headers=headers)
    if resp.status_code == 200:
        table_ids[table_name] = resp.json()["id"]
        print(f"  {table_name}: {table_ids[table_name]}")
    else:
        print(f"  {table_name}: NOT FOUND")

print(f"\nLoaded {len(table_ids)} table IDs.")

In [ ]:
# Add lineage edges representing the FK relationships:
# customers -> orders (customer buys orders)
# orders -> order_items (order contains items)
# products -> order_items (products are ordered)

lineage_edges = [
    {"from": "customers", "to": "orders", "description": "customer_id FK: customer places orders"},
    {"from": "orders", "to": "order_items", "description": "order_id FK: order contains line items"},
    {"from": "products", "to": "order_items", "description": "product_id FK: products appear in order items"}
]

for edge in lineage_edges:
    from_id = table_ids.get(edge["from"])
    to_id = table_ids.get(edge["to"])
    
    if not from_id or not to_id:
        print(f"  Skipping {edge['from']} -> {edge['to']}: table ID not found")
        continue
    
    lineage_payload = {
        "edge": {
            "fromEntity": {
                "id": from_id,
                "type": "table"
            },
            "toEntity": {
                "id": to_id,
                "type": "table"
            },
            "description": edge["description"]
        }
    }
    
    resp = requests.put(f"{OM_URL}/lineage", headers=headers, json=lineage_payload)
    
    if resp.status_code in (200, 201):
        print(f"  Added: {edge['from']} -> {edge['to']}")
    else:
        print(f"  Failed {edge['from']} -> {edge['to']}: {resp.status_code} - {resp.text[:100]}")

print("\nLineage edges created. Check the Lineage tab in the OM UI for visualization.")

## Exercise 5: Impact Analysis (Downstream Dependencies)

**Impact analysis** answers: "If I change this table, what else is affected?"

This is critical for:
- Schema change management (before altering a column, know who depends on it)
- Data quality incidents (a bug in source data -- what downstream is impacted?)
- Compliance (if PII handling changes, what downstream consumers need updating?)

Let's check the downstream impact of the `customers` table.

In [ ]:
# Get lineage with downstream depth
fqn = f"{FQN_PREFIX}.customers"
resp = requests.get(
    f"{OM_URL}/lineage/table/name/{fqn}",
    headers=headers,
    params={"upstreamDepth": 0, "downstreamDepth": 5}
)

if resp.status_code == 200:
    lineage = resp.json()
    entity = lineage.get("entity", {})
    downstream_edges = lineage.get("downstreamEdges", [])
    nodes = {n["id"]: n for n in lineage.get("nodes", [])}
    nodes[entity.get("id", "")] = entity
    
    print(f"Impact Analysis: {entity.get('fullyQualifiedName', 'N/A')}")
    print("=" * 70)
    print(f"\nDirect downstream dependencies: {len(downstream_edges)}")
    
    # Build dependency tree
    affected = set()
    for edge in downstream_edges:
        to_id = edge.get("toEntity", "")
        node = nodes.get(to_id, {})
        node_fqn = node.get("fullyQualifiedName", to_id)
        desc = edge.get("description", "")
        affected.add(node_fqn)
        print(f"  -> {node_fqn}")
        if desc:
            print(f"     Relationship: {desc}")
    
    # Check for indirect downstream (depth > 1)
    all_nodes_fqns = [n.get("fullyQualifiedName", "") for n in nodes.values() if n.get("fullyQualifiedName") != entity.get("fullyQualifiedName")]
    indirect = set(all_nodes_fqns) - affected
    if indirect:
        print(f"\nIndirect downstream (depth > 1):")
        for fqn in indirect:
            if fqn:
                print(f"  -> -> {fqn}")
    
    total_affected = len(affected) + len(indirect)
    print(f"\nTotal affected assets: {total_affected}")
    print(f"\nConclusion: Changes to 'customers' table would impact {total_affected} downstream table(s).")
    print("Before modifying columns, verify compatibility with all downstream consumers.")
else:
    print(f"Lineage request failed: {resp.status_code}")

In [ ]:
# Visualize the complete lineage graph as text
print("E-Commerce Data Lineage Graph")
print("=" * 40)
print()
print("  customers")
print("    |")
print("    +--[customer_id]--> orders")
print("                          |")
print("  products                |")
print("    |                     |")
print("    +--[product_id]---> order_items <--[order_id]--+")
print()
print("Legend:")
print("  --> = FK relationship (data dependency)")
print("  customers: PII data (email, phone)")
print("  orders: transactional data")
print("  order_items: fact table (junction)")
print("  products: reference data")

## Summary

In this notebook, you learned:

1. **Full-text search:** Find any asset by name, description, or metadata content
2. **Tag-based search:** Locate all PII-tagged assets for compliance audits
3. **FK-based lineage:** How foreign key relationships create implicit data lineage
4. **Manual lineage:** Add custom lineage edges for relationships not automatically discovered
5. **Impact analysis:** Assess downstream effects before making changes to upstream tables

**Key concept:** Search makes the catalog usable (finding data), while lineage makes it trustworthy (understanding data flow and dependencies). Together, they enable informed decision-making about data changes.

---

**Congratulations!** You have completed all 4 catalog lab notebooks.

**What you've practiced:**
- Catalog exploration and navigation
- REST API CRUD operations on metadata
- PII tagging and tier classification
- Full-text search and lineage analysis

**Next steps:**
- Explore the OpenMetadata UI at http://localhost:18585 for the visual experience
- Try adding more detailed column-level lineage
- Experiment with glossary terms (`/api/v1/glossaryTerms`)
- When done: `docker compose down` (preserves data) or `docker compose down -v` (clean reset)